# Agrupaciones y Agregación Multinivel de Datos.
Dataset: `pipol_datos.csv`  
Repositorio en GitHub: https://github.com/IsacNM/manejo_datos

In [1]:
import pandas as pd
import datetime

url = 'https://raw.githubusercontent.com/IsacNM/manejo_datos/aeb4be62d829d8b189701164513739c04450380a/pipol_dataset.csv'
df = pd.read_csv(url)
print(df)

        name  last_name                       email    birthday        phone  \
0    Claudia    Chambas        Peinetas69@yahoo.com  04/04/1998  +52 2860679   
1    Jessica    Barrett         michael04@yahoo.com  20/02/1992   +1 4088036   
2      Wendy      Jones       vroberson@hotmail.com  22/12/1980  +34 6878987   
3     Robert   Casillas         pwright@hotmail.com  30/11/1960  +52 3528441   
4     Joseph       Cook   danielskimberly@gmail.com  29/11/1937  +52 7067444   
..       ...        ...                         ...         ...          ...   
95       Ian    Andrade           jesse05@gmail.com  03/10/1988  +34 5654002   
96      Sean     Walker       kenneth50@hotmail.com  03/01/1984  +34 6149841   
97    Angela       Lane       gregory58@hotmail.com  11/04/1988  +52 8053025   
98    Rachel       Page     rachelbrown@hotmail.com  09/12/1987  +54 1768982   
99  Courtney  Patterson  courtneyparker@hotmail.com  06/09/1963  +54 1508734   

           city        country  
0   Me

## Paso 1 — Convertir `birthday` a objeto de tiempo y calcular la edad

In [2]:
# Convertir 'birthday' de string a objeto datetime.
# Las fechas vienen en formato día/mes/año (DD/MM/YYYY).
df['birthday'] = pd.to_datetime(df['birthday'], format='%d/%m/%Y')
print(df.dtypes)
df.head()

name                    str
last_name               str
email                   str
birthday     datetime64[us]
phone                   str
city                    str
country                 str
dtype: object


,name,last_name,email,birthday,phone,city,country
0,Claudia,Chambas,Peinetas69@yahoo.com,1998-04-04,+52 2860679,Mexico City,Mexico
1,Jessica,Barrett,michael04@yahoo.com,1992-02-20,+1 4088036,Los Angeles,United States
2,Wendy,Jones,vroberson@hotmail.com,1980-12-22,+34 6878987,Valencia,Spain
3,Robert,Casillas,pwright@hotmail.com,1960-11-30,+52 3528441,Monterrey,Mexico
4,Joseph,Cook,danielskimberly@gmail.com,1937-11-29,+52 7067444,Mexico City,Mexico


In [3]:
# Calcular la edad de forma vectorizada.
# Restamos la fecha de nacimiento a la fecha actual, dividimos por 365.25 (promedio que incluye años bisiestos) y convertimos a entero.
hoy = pd.Timestamp(datetime.datetime.now())
df['age'] = ((hoy - df['birthday']).dt.days // 365.25).astype(int)

df.head()

,name,last_name,email,birthday,phone,city,country,age
0,Claudia,Chambas,Peinetas69@yahoo.com,1998-04-04,+52 2860679,Mexico City,Mexico,28
1,Jessica,Barrett,michael04@yahoo.com,1992-02-20,+1 4088036,Los Angeles,United States,34
2,Wendy,Jones,vroberson@hotmail.com,1980-12-22,+34 6878987,Valencia,Spain,45
3,Robert,Casillas,pwright@hotmail.com,1960-11-30,+52 3528441,Monterrey,Mexico,65
4,Joseph,Cook,danielskimberly@gmail.com,1937-11-29,+52 7067444,Mexico City,Mexico,88


## Paso 2 — Agrupar personas por país con `groupby`

In [4]:
# Agrupar por país
group = df.groupby('country')

# Ver los grupos disponibles
print(group.groups.keys())

dict_keys(['Argentina', 'Chile', 'Mexico', 'Spain', 'United States'])


In [5]:
# Extraer el grupo de un país específico con get_group.
# Tomamos el primer país disponible en el diccionario de grupos.
primer_pais = list(group.groups.keys())[0]
print(f"Mostrando grupo: {primer_pais}")
group.get_group(primer_pais)

Mostrando grupo: Argentina


,name,last_name,email,birthday,phone,city,country,age
6,Emily,Wilson,susan12@gmail.com,1964-10-01,+54 2108413,Buenos Aires,Argentina,61
15,Paul,Smith,ishannon@gmail.com,1981-03-17,+54 6160052,Rosario,Argentina,45
16,Bianca,Collier,wardjeffery@gmail.com,2001-10-19,+54 7683756,Rosario,Argentina,24
19,John,Blankenship,russelltyrone@gmail.com,1938-12-12,+54 4042670,Córdoba,Argentina,87
21,Calvin,Yates,goodwinjay@hotmail.com,1967-07-04,+54 4997359,Rosario,Argentina,58
22,Lisa,Knight,lewischristy@hotmail.com,1968-04-25,+54 1944282,Rosario,Argentina,58
25,Kathleen,Andrews,vharris@yahoo.com,1971-09-21,+54 3311343,Rosario,Argentina,54
30,Rhonda,Vazquez,lalvarez@yahoo.com,1965-03-21,+54 6897927,Córdoba,Argentina,61
34,Theresa,Stewart,thomas42@gmail.com,1967-01-10,+54 8380234,Córdoba,Argentina,59
36,Stacey,Herrera,monica13@yahoo.com,1955-12-24,+54 7260542,Rosario,Argentina,70


## Paso 3 — Edad promedio por país con `.agg()` y `.mean()`

In [6]:
# Opción 1: con .mean()
edad_promedio = group['age'].mean()
print(edad_promedio)

country
Argentina        56.421053
Chile            50.000000
Mexico           58.040000
Spain            55.571429
United States    61.187500
Name: age, dtype: float64


In [7]:
# Opción 2: con .agg() — más flexible, permite combinar varias métricas
resumen = group['age'].agg(['mean', 'min', 'max', 'count'])
resumen.columns = ['edad_promedio', 'edad_minima', 'edad_maxima', 'total_personas']
resumen

,edad_promedio,edad_minima,edad_maxima,total_personas
country,,,,
Argentina,56.421053,24,91,19
Chile,50.000000,21,87,19
Mexico,58.040000,24,91,25
Spain,55.571429,24,85,21
United States,61.187500,30,92,16


## Paso 4 — Agregación multinivel: agrupar por país y ciudad

In [8]:
# groupby con varias columnas → produce un índice jerárquico (MultiIndex).
# Calculamos varias métricas de 'age' por país y ciudad.
resumen_multinivel = (
    df.groupby(['country', 'city'])['age']
      .agg(edad_promedio='mean',
           edad_minima='min',
           edad_maxima='max',
           total_personas='count')
      .round(2)
)
resumen_multinivel

edad_promedio  edad_minima  edad_maxima  \
country       city                                                    
Argentina     Buenos Aires          69.67           57           91   
              Córdoba               57.67           29           87   
              Rosario               49.14           24           70   
Chile         Concepción            42.80           22           80   
              Santiago              51.43           27           87   
              Valparaíso            53.71           21           82   
Mexico        Dusty City            53.33           24           76   
              Mexico City           56.86           25           88   
              Monterrey             63.67           38           91   
Spain         Barcelona             63.89           38           85   
              Madrid                47.38           24           84   
              Valencia              53.25           39           82   
United States Chicago               73.50           55           92   
              Los Angeles           56.86           30           86   
              New York              62.00           38           91   

                            total_personas  
country       city                          
Argentina     Buenos Aires               3  
              Córdoba                    9  
              Rosario                    7  
Chile         Concepción                 5  
              Santiago                   7  
              Valparaíso                 7  
Mexico        Dusty City                 9  
              Mexico City                7  
              Monterrey                  9  
Spain         Barcelona                  9  
              Madrid                     8  
              Valencia                   4  
United States Chicago                    2  
              Los Angeles                7  
              New York                   7